<a href="https://colab.research.google.com/github/anthony-yt/deepwater-image-segmentation-classical-quantum/blob/feat%2Fuwsegformer-inference/notebooks/3_uwsegformer_inference_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# UWSegFormer Offline Inference Test

Qualitative comparison of UWSegFormer predictions using checkpoints pretrained on SUIM and DUT over 5 internal project images.

## 1. Colab Environment Bootstrap

Run this section in Colab. The legacy UWSegFormer stack is installed into a separate conda environment named `uws` with Python 3.8, because current Colab Python versions are too new for `mmcv-full==1.5.3`.

In [ ]:
# Colab only: install conda support. This restarts the runtime once.
try:
    import google.colab  # noqa: F401
    import condacolab  # noqa: F401
    print('condacolab is already installed.')
except ImportError:
    try:
        import google.colab  # noqa: F401
        !pip install -q condacolab
        import condacolab
        condacolab.install()
    except ImportError:
        print('Not running in Colab; skipping condacolab installation.')


In [ ]:
# Colab only: create the legacy Python 3.8 environment and install UWSegFormer dependencies.
# If this cell has already completed successfully, it is safe to skip it on later runs.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    !conda create -n uws python=3.8 -y
    !conda run -n uws python -m pip install --upgrade pip
    !conda run -n uws python -m pip install torch==1.10.2+cu111 torchvision==0.11.3+cu111 -f https://download.pytorch.org/whl/torch_stable.html
    !conda run -n uws python -m pip install mmcv-full==1.5.3 -f https://download.openmmlab.com/mmcv/dist/cu111/torch1.10/index.html
    !conda run -n uws python -m pip install mmsegmentation==0.11.0 opencv-python matplotlib pandas pillow scipy
    !conda run -n uws python -m pip install scipy
    !conda run -n uws python -m pip install timm
    !conda run -n uws python -m pip install scipy timm==0.4.12 kornia terminaltables yapf addict packaging
else:
    print('Local run detected. Use your local environment instead of this Colab setup cell.')


## 2. Paths and External Code

In [ ]:
import os
import tempfile
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', tempfile.mkdtemp(prefix='matplotlib-'))

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive', force_remount=True)
except Exception:
    print('Not running in Colab or Drive is already mounted.')


In [ ]:
from pathlib import Path
import pandas as pd

DRIVE_ROOT = Path('/content/drive/MyDrive/Under_Water_Dataset')

PROJECT_TEST_DIR = DRIVE_ROOT / 'Dataset_Processed' / 'Dataset_Final_512'
UWSEGFORMER_ROOT = Path('/content/UWSegFormer')
CHECKPOINT_ROOT = DRIVE_ROOT / 'checkpoints'

SUIM_CONFIG = UWSEGFORMER_ROOT / 'local_configs/uwsegformer/segformer_e+PPsegHead1.b0.640x480.suim.160k.py'
DUT_CONFIG = UWSEGFORMER_ROOT / 'local_configs/uwsegformer/segformer_e+PPsegHead1.b0.640x480.dut.40k.py'

SUIM_CHECKPOINT = CHECKPOINT_ROOT / 'uws_SUIM.pth'
DUT_CHECKPOINT = CHECKPOINT_ROOT / 'uws_dut.pth'

pd.Series({
    'PROJECT_TEST_DIR': PROJECT_TEST_DIR,
    'SUIM_CONFIG': SUIM_CONFIG,
    'DUT_CONFIG': DUT_CONFIG,
    'SUIM_CHECKPOINT': SUIM_CHECKPOINT,
    'DUT_CHECKPOINT': DUT_CHECKPOINT,
})

In [ ]:
# Colab only: clone UWSegFormer source if it is not already present.
if IN_COLAB and not UWSEGFORMER_ROOT.exists():
    !git clone https://github.com/SAWRJJ/UWSegFormer.git /content/UWSegFormer
else:
    print(f'UWSegFormer source already available or local run: {UWSEGFORMER_ROOT}')

assert SUIM_CONFIG.exists(), f'Missing SUIM config: {SUIM_CONFIG}'
assert DUT_CONFIG.exists(), f'Missing DUT config: {DUT_CONFIG}'

In [ ]:
# Colab only: patch a stale import in the upstream UWSegFormer repo.
# The configs use MAA_segformer1, but decode_heads/__init__.py imports old class names
# that are not defined in segformer_head.py.
decode_heads_init = UWSEGFORMER_ROOT / 'mmseg' / 'models' / 'decode_heads' / '__init__.py'
if decode_heads_init.exists():
    text = decode_heads_init.read_text()
    text = text.replace(
        'from .segformer_head import PPMobileSegHead_segformer2,PPMobileSegHead_segformer1',
        'from .segformer_head import MAA_segformer1',
    )
    text = text.replace(
        "    'PPMobileSegHead_segformer2',\n    'PPMobileSegHead_segformer1'",
        "    'MAA_segformer1'",
    )
    decode_heads_init.write_text(text)
    print(f'Patched {decode_heads_init}')
else:
    print(f'Missing file to patch: {decode_heads_init}')


In [ ]:
for config_path in [
    Path('/content/UWSegFormer/local_configs/uwsegformer/segformer_e+PPsegHead1.b0.640x480.suim.160k.py'),
    Path('/content/UWSegFormer/local_configs/uwsegformer/segformer_e+PPsegHead1.b0.640x480.dut.40k.py'),
]:
    text = config_path.read_text()
    text = text.replace('../../_base_/', '../_base_/')
    config_path.write_text(text)
    print("patched", config_path)

## 3. Select 5 Test Images

For the qualitative comparison, use 5 project images. If the Drive folder is unavailable, upload 5 images manually or override `PROJECT_TEST_DIR`.

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}

candidate_dirs = [
    PROJECT_TEST_DIR,
    DRIVE_ROOT / 'Dataset_Processed' / 'Dataset_Final_512',
    DRIVE_ROOT / 'Dataset_Processed' / 'Frames_Crudos',
    DRIVE_ROOT / 'Dataset_Processed' / 'Frames_CLAHE',
    DRIVE_ROOT / 'Dataset_Processed' / 'Frames_MSRCR',
    DRIVE_ROOT / 'Dataset_Processed' / 'Frames_UDCP',
    # REPO_ROOT / 'data' / 'test_images', # Removed as REPO_ROOT is not defined in Colab
    # REPO_ROOT / 'data' / 'SUIM' / 'images' / 'validation', # Removed as REPO_ROOT is not defined in Colab
]

available_dirs = []
for directory in candidate_dirs:
    if directory.exists():
        image_count = sum(
            1 for path in directory.rglob('*')
            if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS and 'mask' not in path.name.lower()
        )
        available_dirs.append({'directory': str(directory), 'image_count': image_count})

if available_dirs:
    display(pd.DataFrame(available_dirs))
else:
    print('No candidate image directories were found. Check Drive mount, folder names, or upload 5 images.')

if not PROJECT_TEST_DIR.exists() or not any(PROJECT_TEST_DIR.rglob('*')):
    non_empty_dirs = [Path(row['directory']) for row in available_dirs if row['image_count'] > 0]
    if non_empty_dirs:
        PROJECT_TEST_DIR = non_empty_dirs[0]
        print(f'Using fallback PROJECT_TEST_DIR: {PROJECT_TEST_DIR}')

candidate_images = sorted(
    path for path in PROJECT_TEST_DIR.rglob('*')
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS and 'mask' not in path.name.lower()
)

test_images = candidate_images[:5]
if not test_images:
    raise FileNotFoundError(
        f'No test images found in {PROJECT_TEST_DIR}. '
        'Set PROJECT_TEST_DIR to a folder that contains images, or upload/select 5 images first.'
    )

pd.DataFrame({'image_path': [str(path) for path in test_images]})

In [ ]:
# Optional fallback: upload exactly the 5 qualitative test images manually.
# Run this only if the previous cell cannot find images in Drive.
# from google.colab import files
# uploaded = files.upload()
# PROJECT_TEST_DIR = Path('/content/test_images')
# PROJECT_TEST_DIR.mkdir(exist_ok=True)
# for name, data in uploaded.items():
#     (PROJECT_TEST_DIR / name).write_bytes(data)
# test_images = sorted(PROJECT_TEST_DIR.iterdir())[:5]
# pd.DataFrame({'image_path': [str(path) for path in test_images]})


## 4. Load Models

In [ ]:
def load_mmseg_model(config_path: Path, checkpoint_path: Path, device: str = 'cuda:0'):
    from mmseg.apis import init_segmentor

    if not config_path.exists():
        raise FileNotFoundError(f'Missing config: {config_path}')
    if not checkpoint_path.exists():
        raise FileNotFoundError(f'Missing checkpoint: {checkpoint_path}')
    return init_segmentor(str(config_path), str(checkpoint_path), device=device)


def run_mmseg_inference(model, image_path: Path) -> np.ndarray:
    from mmseg.apis import inference_segmentor

    result = inference_segmentor(model, str(image_path))
    pred = result[0] if isinstance(result, (list, tuple)) else result
    return pred.astype(np.uint8)


DEVICE = 'cuda:0'

if IN_COLAB:
    print('Model loading should be executed inside the conda env via the script cell below.')
else:
    suim_model = load_mmseg_model(SUIM_CONFIG, SUIM_CHECKPOINT, DEVICE)
    dut_model = load_mmseg_model(DUT_CONFIG, DUT_CHECKPOINT, DEVICE)


## 5. Visual Comparison

In [ ]:
def read_rgb(image_path: Path) -> np.ndarray:
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError(f'Could not read image: {image_path}')
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


def colorize_mask(mask: np.ndarray) -> np.ndarray:
    mask = mask.astype(np.uint8)
    colored = cv2.applyColorMap((mask * 30) % 255, cv2.COLORMAP_TURBO)
    return cv2.cvtColor(colored, cv2.COLOR_BGR2RGB)


def plot_comparison(image_path: Path, suim_mask: np.ndarray, dut_mask: np.ndarray):
    image = read_rgb(image_path)
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(image)
    axes[0].set_title('Original')
    axes[1].imshow(colorize_mask(suim_mask))
    axes[1].set_title('UWSegFormer - SUIM weights')
    axes[2].imshow(colorize_mask(dut_mask))
    axes[2].set_title('UWSegFormer - DUT weights')
    for ax in axes:
        ax.axis('off')
    plt.tight_layout()
    plt.show()


In [ ]:
if not IN_COLAB:
    for image_path in test_images:
        suim_prediction = run_mmseg_inference(suim_model, image_path)
        dut_prediction = run_mmseg_inference(dut_model, image_path)
        plot_comparison(image_path, suim_prediction, dut_prediction)
else:
    print('In Colab, run inference from the conda env and then load generated outputs in the next section.')


## 6. Colab Conda Inference Script

This section runs UWSegFormer with the `uws` conda environment and writes prediction masks to `/content/uwsegformer_outputs`.

In [ ]:
from pathlib import Path

output_dir = Path('/content/uwsegformer_outputs')
output_dir.mkdir(exist_ok=True)

image_list_path = Path('/content/test_image_paths.txt')
image_list_path.write_text('\n'.join(str(path) for path in test_images))

script_path = Path('/content/run_uwsegformer_inference.py')
script_path.write_text(f'''
import sys
from pathlib import Path

import cv2
import numpy as np

UWSEGFORMER_ROOT = Path(r"{UWSEGFORMER_ROOT}")
sys.path.insert(0, str(UWSEGFORMER_ROOT))

from mmseg.apis import init_segmentor, inference_segmentor

IMAGE_LIST = Path(r"{image_list_path}")
OUTPUT_DIR = Path(r"{output_dir}")
RESIZED_DIR = OUTPUT_DIR / "resized_inputs"
RESIZED_DIR.mkdir(exist_ok=True)

SUIM_CONFIG = Path(r"{SUIM_CONFIG}")
DUT_CONFIG = Path(r"{DUT_CONFIG}")
SUIM_CHECKPOINT = Path(r"{SUIM_CHECKPOINT}")
DUT_CHECKPOINT = Path(r"{DUT_CHECKPOINT}")

DEVICE = "cuda:0"

suim_model = init_segmentor(str(SUIM_CONFIG), str(SUIM_CHECKPOINT), device=DEVICE)
dut_model = init_segmentor(str(DUT_CONFIG), str(DUT_CHECKPOINT), device=DEVICE)

for image_path in IMAGE_LIST.read_text().splitlines():
    image_path = Path(image_path)
    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        print(f"skipping unreadable image {{image_path}}")
        continue

    resized_path = RESIZED_DIR / f"{{image_path.stem}}_640x480{{image_path.suffix}}"
    resized = cv2.resize(image, (640, 480), interpolation=cv2.INTER_AREA)
    cv2.imwrite(str(resized_path), resized)

    suim_pred = inference_segmentor(suim_model, str(resized_path))[0].astype(np.uint8)
    dut_pred = inference_segmentor(dut_model, str(resized_path))[0].astype(np.uint8)

    cv2.imwrite(str(OUTPUT_DIR / f"{{image_path.stem}}_suim.png"), suim_pred)
    cv2.imwrite(str(OUTPUT_DIR / f"{{image_path.stem}}_dut.png"), dut_pred)

    print(f"wrote predictions for {{image_path.name}}")
''')

print(script_path)

In [ ]:
# Colab only: run the generated script in the legacy conda environment.
if IN_COLAB:
    !conda run -n uws python /content/run_uwsegformer_inference.py
else:
    print('Local run detected; skip this Colab-only command cell.')


In [ ]:
# Colab only: load saved masks and show the comparison grids in this notebook.
if IN_COLAB:
    output_dir = Path('/content/uwsegformer_outputs')
    for image_path in test_images:
        suim_mask = cv2.imread(str(output_dir / f'{image_path.stem}_suim.png'), cv2.IMREAD_GRAYSCALE)
        dut_mask = cv2.imread(str(output_dir / f'{image_path.stem}_dut.png'), cv2.IMREAD_GRAYSCALE)
        if suim_mask is None or dut_mask is None:
            print(f'Missing prediction outputs for {image_path.name}')
            continue
        plot_comparison(image_path, suim_mask, dut_mask)
else:
    print('Local run detected; comparison was already plotted above.')


## 7. Qualitative Notes

| Image | SUIM checkpoint observation | DUT checkpoint observation | Preferred |
|---|---|---|---|
| 1 | TBD | TBD | TBD |
| 2 | TBD | TBD | TBD |
| 3 | TBD | TBD | TBD |
| 4 | TBD | TBD | TBD |
| 5 | TBD | TBD | TBD |